# GTU Mimari Lejant - Hybrid YOLO + SAM2

Bu notebook hibrit hatti test eder: YOLO sinif/bbox uretir, akilli filtre false positive adaylari azaltir, SAM2 sadece kalan kutulari maskeler.

In [ ]:
!nvidia-smi
import torch
print('CUDA available:', torch.cuda.is_available())

## 1. Repo'yu Klonla

In [ ]:
%cd /content
!rm -rf /content/lejanter_doga_vlm_codex
!git clone https://github.com/doganalci/lejanter_doga_vlm_codex.git /content/lejanter_doga_vlm_codex
%cd /content/lejanter_doga_vlm_codex
!pip install -q -r requirements.txt

## 2. Drive'dan Weight ve Test Gorsellerini Al

In [ ]:
from pathlib import Path
import shutil

SHARED_DRIVE_FOLDER_URL = 'https://drive.google.com/drive/folders/1vmSSPsnu4fVBkM0yn1ScU7DFgDXyOksa?usp=sharing'
SHARED_DOWNLOAD_DIR = Path('/content/shared_drive_lejant')
if SHARED_DOWNLOAD_DIR.exists():
    shutil.rmtree(SHARED_DOWNLOAD_DIR)
SHARED_DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

!pip install -q gdown
!gdown --folder '{SHARED_DRIVE_FOLDER_URL}' -O /content/shared_drive_lejant --remaining-ok

PROJECT_DIR = Path('/content/lejanter_doga_vlm_codex')
weights_src = SHARED_DOWNLOAD_DIR / 'weights'
if not weights_src.exists() and any(SHARED_DOWNLOAD_DIR.glob('*.pt')):
    weights_src = SHARED_DOWNLOAD_DIR
weights_dst = PROJECT_DIR / 'drive_weights'
if weights_dst.exists():
    shutil.rmtree(weights_dst)
weights_dst.mkdir(parents=True, exist_ok=True)
if weights_src.exists():
    for pt in weights_src.rglob('*.pt'):
        shutil.copy2(pt, weights_dst / pt.name)
print('Weights:')
for pt in sorted(weights_dst.glob('*.pt')):
    print('-', pt)

test_src = SHARED_DOWNLOAD_DIR / 'test'
if not test_src.exists():
    has_root_images = any(p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tif', '.tiff', '.avif'} for p in SHARED_DOWNLOAD_DIR.iterdir())
    if has_root_images:
        test_src = SHARED_DOWNLOAD_DIR
test_dst = PROJECT_DIR / 'data/raw/drive_test'
if test_dst.exists():
    shutil.rmtree(test_dst)
test_dst.mkdir(parents=True, exist_ok=True)
if test_src.exists():
    for image in test_src.rglob('*'):
        if image.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tif', '.tiff', '.avif'}:
            shutil.copy2(image, test_dst / image.name)
print('Test images:')
for image in sorted(test_dst.glob('*')):
    print('-', image)

## 3. YOLO Raw Proposal Uret

Burada conf dusuk tutulur; daha sonra filtre uygulayacagiz. Boylece filtreleme etkisini kontrollu goruruz.

In [ ]:
from pathlib import Path

PROJECT_DIR = Path('/content/lejanter_doga_vlm_codex')
test_source = PROJECT_DIR / 'data/raw/drive_test'
assert test_source.exists() and any(test_source.iterdir()), 'Drive test gorseli bulunamadi.'

weights_dir = PROJECT_DIR / 'drive_weights'
candidates = list(weights_dir.glob('elements-seg-v3-no-erasing-best.pt')) or list(weights_dir.glob('*v3*best*.pt')) or list(weights_dir.glob('*.pt'))
assert candidates, 'Weight dosyasi bulunamadi.'
weight_path = candidates[0]
print('Using:', weight_path)

!cd /content/lejanter_doga_vlm_codex && python scripts/infer_yolo.py \
  --weights {str(weight_path)} \
  --source {str(test_source)} \
  --task segment \
  --conf 0.25 \
  --out /content/lejanter_doga_vlm_codex/outputs/hybrid/yolo_raw_conf025.json \
  --name hybrid-yolo-raw-conf025 \
  --project /content/lejanter_doga_vlm_codex/outputs/runs \
  --save-visuals \
  --device 0

## 4. Akilli Filtre Uygula

Ilk deneme icin cam ve ahsap_dograma daha yuksek threshold aliyor. Gerekirse burada thresholdlari degistirecegiz.

In [ ]:
!cd /content/lejanter_doga_vlm_codex && python scripts/filter_detections.py \
  --input /content/lejanter_doga_vlm_codex/outputs/hybrid/yolo_raw_conf025.json \
  --out /content/lejanter_doga_vlm_codex/outputs/hybrid/yolo_filtered_conf070.json \
  --default-conf 0.70 \
  --class-threshold cam=0.75 \
  --class-threshold ahsap_dograma=0.70 \
  --class-threshold camur_harc=0.60 \
  --min-area-ratio 0.00005 \
  --max-area-ratio 0.35 \
  --nms-iou 0.35

## 5. SAM2 Kurulumu

In [ ]:
%cd /content
!rm -rf /content/sam2
!git clone https://github.com/facebookresearch/sam2.git /content/sam2
%cd /content/sam2
!pip install -q -e .
!mkdir -p /content/sam2/checkpoints
!wget -q -nc -O /content/sam2/checkpoints/sam2.1_hiera_tiny.pt https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_tiny.pt
%cd /content/lejanter_doga_vlm_codex
print('SAM2 ready')

## 6. Filtrelenmis YOLO Kutularini SAM2 ile Refine Et

In [ ]:
!cd /content/sam2 && python /content/lejanter_doga_vlm_codex/scripts/refine_with_sam2.py \
  --detections /content/lejanter_doga_vlm_codex/outputs/hybrid/yolo_filtered_conf070.json \
  --checkpoint /content/sam2/checkpoints/sam2.1_hiera_tiny.pt \
  --model-cfg configs/sam2.1/sam2.1_hiera_t.yaml \
  --out /content/lejanter_doga_vlm_codex/outputs/hybrid/hybrid_yolo_conf070_sam2.json \
  --visual-dir /content/lejanter_doga_vlm_codex/outputs/hybrid/visuals_hybrid_conf070_sam2 \
  --device cuda

!find /content/lejanter_doga_vlm_codex/outputs/hybrid -maxdepth 2 -type f | sort | sed -n '1,160p'

## 7. Sonuclari Goster

In [ ]:
from IPython.display import Image, display
from pathlib import Path

visual_dir = Path('/content/lejanter_doga_vlm_codex/outputs/hybrid/visuals_hybrid_conf070_sam2')
for image_path in sorted(visual_dir.glob('*'))[:10]:
    if image_path.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.avif'}:
        display(Image(filename=str(image_path)))

## 8. Drive Reports Klasorune Kopyala

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil
import datetime

drive.mount('/content/drive')
REPORTS_DIR = Path('/content/drive/MyDrive/_doganalci_onedrive/codes_doga_doktora_2025/lejant_vllm_sehemntaton_report_may_2026/reports')
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
export_dir = REPORTS_DIR / f'hybrid_yolo_sam2_{stamp}'
shutil.copytree('/content/lejanter_doga_vlm_codex/outputs/hybrid', export_dir, dirs_exist_ok=True)
print('Copied to:', export_dir)